# Synthetic valley: a simple PEST interface for the steady state

This notebook is the companion of [`synthetic-valley-c-manual-calibration`](synthetic-valley-c-manual-calibration.ipynb). Here we will calibrate the same four parameters with PEST++ instead of to the sliders: the K above the confining unit, the K below it, the vertical K of the confining unit, and a recharge multiplier, calibrated to the same 34 steady-state heads, with the same prediction, the steady-state discharge of groundwater to the stream.

The interface is built directly against the model's own files: **template files** that are copies of MODFLOW 6 input files with the parameters marked in them, and **instruction files** that tell PEST++ where to read the simulated heads in the output files - this is a simple and clean PEST interface. We use pyEMU to assemble the control file from them. PESTPP-IES, the iterative ensemble smoother, then calibrates an ensemble of parameter sets drawn from the prior, with parameter bounds and weights that you choose, so the result shows not one best fit but the range of parameters that fit the heads.

## Imports

`sv_notebook_helpers` holds the location of the base model and `true_observations()`.

In [ ]:
%matplotlib inline
import os
import pathlib as pl

import flopy
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu
from sv_notebook_helpers import BASE_MODEL, true_observations

## The steady-state model

The model is cut down to its steady-state stress period as in the companion notebook, and its properties are made constant in each layer in the same way: K, `k33`, specific storage, and specific yield each become one value per layer, the geometric mean of the layer's active cells, which MODFLOW 6 reads as a `CONSTANT`. As in the companion notebook, this removes the lake, whose very high K in layer 1 cannot be one value of the layer. The NPF file is then short, and its template file is too.

The recharge is the other parameter. It is changed from a list of cells to an array, `READASARRAYS`, holding the same rates, and `set_all_data_external()` writes the array to its own file, which the RCH file reads with a line like `OPEN/CLOSE 'sv.rch_recharge_1.txt' FACTOR 1.0`. MODFLOW 6 multiplies every value of the array by its `FACTOR`, so that one number scales the recharge of every cell.

In [ ]:
name = "sv"
ws = pl.Path("models/sv-steady")  # the steady-state model and its PEST interface

sim = flopy.mf6.MFSimulation.load(
    sim_name=name, sim_ws=BASE_MODEL, write_headers=False, verbosity_level=0
)
sim.set_sim_path(ws)
gwf = sim.get_model(name)

# one stress period: the steady state
steady_state = sim.tdis.perioddata.array[0]
sim.tdis.nper = 1
sim.tdis.perioddata = [tuple(steady_state)]

# the wells only pump after the steady state
gwf.remove_package("pwell")
gwf.remove_package("prediction")

# evapotranspiration of stress period 1 only
nper = len(gwf.evt.stress_period_data.get_data())
for kper in range(1, nper):
    gwf.evt.stress_period_data.remove_transient_key(kper)

# one value per layer of each property: the geometric mean of the layer's
# active cells, leaving out the lake cells of layer 1
aquifer = gwf.dis.idomain.array > 0
lake = gwf.npf.k.array[0] > 1000.0
aquifer[0] = aquifer[0] & ~lake


def layer_means(values):
    return [np.exp(np.log(values[k][aquifer[k]]).mean()) for k in range(len(values))]


k_layers = layer_means(gwf.npf.k.array)
k33_layers = layer_means(gwf.npf.k33.array)
gwf.npf.k.set_data(k_layers)
gwf.npf.k33.set_data(k33_layers)
gwf.sto.ss.set_data(layer_means(gwf.sto.ss.array))
gwf.sto.sy.set_data(layer_means(gwf.sto.sy.array))

# the recharge of stress period 1 as an array, which a FACTOR can multiply
nrow, ncol = gwf.dis.nrow.data, gwf.dis.ncol.data
recharge = np.zeros((nrow, ncol))
rch_list = gwf.rch.stress_period_data.get_data(0)
for cellid, rate in zip(rch_list["cellid"], rch_list["recharge"]):
    recharge[cellid[1], cellid[2]] = rate  # every recharge cell is in layer 1
gwf.remove_package("rch")
flopy.mf6.ModflowGwfrcha(gwf, recharge={0: recharge}, pname="rch", filename="sv.rch")

# a looser closure than the model's, 0.0001 m, saves iterations and changes
# heads by far less than the noise
ims = sim.get_package("ims")
ims.complexity = "moderate"
ims.outer_dvclose = 1.0e-4
ims.outer_maximum = 500
ims.inner_dvclose = 1.0e-5

sim.set_all_data_external(check_data=False)
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print((ws / "sv.npf").read_text())
print((ws / "sv.rch").read_text())

## Template files

A template file is a copy of a model input file in which each value that a parameter sets is replaced by the parameter's name between two marker characters, here `~`. Its first line, `ptf ~`, names the marker. Before each model run, PEST++ writes the parameter's value into the space between the markers.

The NPF template is the NPF file above, written out in full, with four constants replaced:

- `kupper`, the K of layers 1 and 2, above the confining unit,
- `klower`, the K of layers 4 and 5, below it, and
- `kv`, the `k33` of layer 3. The NPF package reads `k33` as the ratio of vertical to horizontal K (`K33OVERK`), so the parameter's `scale`, set below, divides the vertical K by the horizontal K of layer 3: PEST++ writes the parameter value times its scale.

The RCH template is the RCH file with `rch` in place of the `FACTOR` of the recharge array.

In [ ]:
k3 = k_layers[2]  # the horizontal K of the confining unit, which stays as it is
npf_tpl = f"""ptf ~
BEGIN options
  SAVE_SPECIFIC_DISCHARGE
  K33OVERK
END options

BEGIN griddata
  icelltype
    CONSTANT  0
  k  LAYERED
    CONSTANT  ~ kupper   ~
    CONSTANT  ~ kupper   ~
    CONSTANT  {k3:.8e}
    CONSTANT  ~ klower   ~
    CONSTANT  ~ klower   ~
  k33  LAYERED
    CONSTANT  {k33_layers[0]:.8e}
    CONSTANT  {k33_layers[1]:.8e}
    CONSTANT  ~ kv       ~
    CONSTANT  {k33_layers[3]:.8e}
    CONSTANT  {k33_layers[4]:.8e}
END griddata
"""
(ws / "sv.npf.tpl").write_text(npf_tpl)

rch_tpl = """ptf ~
BEGIN options
  READASARRAYS
END options

BEGIN period  1
  recharge
    OPEN/CLOSE  'sv.rch_recharge_1.txt'  FACTOR  ~ rch      ~
END period  1
"""
(ws / "sv.rch.tpl").write_text(rch_tpl)
print(npf_tpl)

## Instruction files

An instruction file tells PEST++ how to read the simulated values from a model output file. Its first line, `pif ~`, names the marker. The heads are in `sv.gwf.wt.csv` and `sv.gwf.aq.csv`, a header line and then one line of values for the steady state, separated by commas. `l1` moves down one line, so the first `l1` skips the header and the second moves to the values; then each `~,~` moves past the next comma and `!name!` reads the number after it as the observation `name`.

The **prediction** is read the same way: `sv.riv.obs.csv` holds the steady-state flow between the aquifer and the stream, `RIV-SWGW`, negative when groundwater discharges to the stream. It becomes the observation `riv-swgw`, which gets no weight, so IES reports it in every realization but does not fit it.

In [ ]:
head_files = ["sv.gwf.wt.csv", "sv.gwf.aq.csv"]
for csv_file in head_files:
    columns = pd.read_csv(ws / csv_file, index_col=0).columns
    values_line = "l1"
    for column in columns:
        values_line += f" ~,~ !{column.lower()}!"
    lines = ["pif ~", "l1", values_line]
    (ws / (csv_file + ".ins")).write_text("\n".join(lines) + "\n")

# the prediction: the steady-state flow between the aquifer and the stream
lines = ["pif ~", "l1", "l1 ~,~ !riv-swgw!"]
(ws / "sv.riv.obs.csv.ins").write_text("\n".join(lines) + "\n")
output_files = head_files + ["sv.riv.obs.csv"]
print((ws / "sv.gwf.wt.csv.ins").read_text()[:200])
print((ws / "sv.riv.obs.csv.ins").read_text())

## The control file

`pyemu.Pst.from_io_files()` reads the template and instruction files, finds the parameter and observation names in them, and builds a control file around them, with default values to be replaced:

- **Parameters.** Each gets its starting value, the same as the companion notebook's, and is log transformed, because K and recharge are uncertain by factors. `kv` gets the scale that turns it into the ratio MODFLOW reads. The bounds are set with the sliders below.
- **Observations.** The observed values are the true steady-state heads, in the groups `wt` and `aq`, with a weight of 10, the inverse of a measurement standard deviation of 0.1 m. The prediction `riv-swgw` gets the truth's steady-state flow and a weight of 0, and the option `forecasts` tells PEST++ that it is a prediction.
- **Control.** The model command is `mf6`.

In [ ]:
tpl_files = ["sv.npf.tpl", "sv.rch.tpl"]
in_files = ["sv.npf", "sv.rch"]
ins_files = [f + ".ins" for f in output_files]
pst = pyemu.Pst.from_io_files(
    [str(ws / f) for f in tpl_files],
    in_files,
    [str(ws / f) for f in ins_files],
    output_files,
    pst_path=".",
)

# the starting values of the companion notebook
start_k_upper = np.sqrt(k_layers[0] * k_layers[1])  # their geometric mean
start_k_lower = np.sqrt(k_layers[3] * k_layers[4])
start_kv = k33_layers[2] * k3

par = pst.parameter_data
par["partrans"] = "log"
par["pargp"] = par.parnme
par.loc["kupper", ["parval1", "parlbnd", "parubnd"]] = [start_k_upper, 1.0, 1000.0]
par.loc["klower", ["parval1", "parlbnd", "parubnd"]] = [start_k_lower, 1.0, 1000.0]
par.loc["kv", ["parval1", "parlbnd", "parubnd"]] = [start_kv, 1.0e-5, 1.0]
par.loc["kv", "scale"] = 1.0 / k3  # MODFLOW reads the ratio kv / k3
par.loc["rch", ["parval1", "parlbnd", "parubnd"]] = [1.0, 0.2, 5.0]

truth = true_observations()
t_steady = truth.index[0]  # the end of the steady-state stress period, days
obs = pst.observation_data
obs["obsval"] = truth.loc[t_steady, obs.obsnme].values
obs["obgnme"] = obs.obsnme.str[:2]  # wt or aq
obs["weight"] = 10.0
heads = obs.obgnme.isin(["wt", "aq"])  # the history; the rest is the prediction
prediction = "riv-swgw"
obs.loc[prediction, "obgnme"] = "pred"
obs.loc[prediction, "weight"] = 0.0
pst.pestpp_options["forecasts"] = prediction
print(f"true steady-state discharge to the stream {-obs.obsval[prediction]:.0f} m3/d")

pst.model_command = ["mf6"]
pst.write(ws / "pest.pst", version=2)
par[["parval1", "parlbnd", "parubnd", "scale", "partrans"]]

## A test run

`noptmax=0` makes PEST++ run the model once with the starting values: it writes the input files from the templates, runs MODFLOW 6, and reads the heads with the instructions. Its phi should be that of the starting sliders of the companion notebook.

In [ ]:
pst.control_data.noptmax = 0
pst.write(ws / "pest.pst", version=2)
pyemu.os_utils.run("pestpp-ies pest.pst", cwd=ws)
pst.set_res(str(ws / "pest.base.rei"))
print(f"phi of the starting values {pst.phi:.1f} for {pst.nnz_obs} observations")

## Choose the prior and the weights

Each parameter's prior has a **mean** and two **bounds**. The prior ensemble is drawn around the mean, with a standard deviation of a quarter of the distance between the bounds, in log space, and IES keeps every realization inside the bounds. The mean is also the parameter's value in the control file, the one the `base` realization starts from. The weights say how much each group of heads counts in phi: a weight of 10 counts a residual of 0.1 m as 1, and a weight of 0 leaves the group out of the calibration, although PEST++ still reports it.

The sliders set the mean, the lower bound, and the upper bound of each parameter, on a logarithmic scale, and the weight of each group. They start at the values of the test run. The plot shows each parameter's bounds, with its mean as a dot; a mean outside its bounds is moved to the nearest bound.

In [ ]:
def log_slider(value, low, high, description):
    return widgets.FloatLogSlider(
        value=value,
        min=low,
        max=high,
        step=0.05,
        description=description,
        style={"description_width": "160px"},
        layout=widgets.Layout(width="420px"),
    )


def weight_slider(description):
    return widgets.FloatSlider(
        value=10.0,
        min=0.0,
        max=20.0,
        step=1.0,
        description=description,
        style={"description_width": "160px"},
        layout=widgets.Layout(width="420px"),
    )


controls = {
    "kupper_mean": log_slider(start_k_upper, -1.0, 4.0, "kupper mean (m/d)"),
    "kupper_low": log_slider(1.0, -1.0, 4.0, "kupper lower (m/d)"),
    "kupper_high": log_slider(1000.0, -1.0, 4.0, "kupper upper (m/d)"),
    "klower_mean": log_slider(start_k_lower, -1.0, 4.0, "klower mean (m/d)"),
    "klower_low": log_slider(1.0, -1.0, 4.0, "klower lower (m/d)"),
    "klower_high": log_slider(1000.0, -1.0, 4.0, "klower upper (m/d)"),
    "kv_mean": log_slider(start_kv, -7.0, 1.0, "kv mean (m/d)"),
    "kv_low": log_slider(1.0e-5, -7.0, 1.0, "kv lower (m/d)"),
    "kv_high": log_slider(1.0, -7.0, 1.0, "kv upper (m/d)"),
    "rch_mean": log_slider(1.0, -2.0, 2.0, "rch mean"),
    "rch_low": log_slider(0.2, -2.0, 2.0, "rch lower"),
    "rch_high": log_slider(5.0, -2.0, 2.0, "rch upper"),
    "wt_weight": weight_slider("weight of wt heads"),
    "aq_weight": weight_slider("weight of aq heads"),
}


def show_prior(**values):
    fig, axes = plt.subplots(4, 1, figsize=(6.0, 3.6), layout="constrained")
    for ax, name in zip(axes, ["kupper", "klower", "kv", "rch"]):
        mean = values[f"{name}_mean"]
        low, high = values[f"{name}_low"], values[f"{name}_high"]
        ax.plot([low, high], [0, 0], color="tab:blue", lw=6, solid_capstyle="butt")
        ax.plot(np.clip(mean, low, high), 0, "o", color="black")
        ax.set_xscale("log")
        # the full range of the sliders, so the bounds and the mean move against it
        slider = controls[f"{name}_mean"]
        ax.set_xlim(10.0**slider.min, 10.0**slider.max)
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")
        if low >= high:
            ax.set_title("the lower bound must be below the upper bound", color="red")
        elif not low <= mean <= high:
            ax.set_title("the mean is outside the bounds", color="red")
    fig.suptitle(f"weights: wt {values['wt_weight']:.0f}, aq {values['aq_weight']:.0f}")
    plt.show()


display(
    widgets.HBox(
        [
            widgets.VBox(list(controls.values())[:6]),
            widgets.VBox(list(controls.values())[6:12]),
        ]
    ),
    widgets.HBox(list(controls.values())[12:]),
    widgets.interactive_output(show_prior, controls),
)

## Calibrate with PEST++ IES

The next cell takes the final values of the sliders, writes them into the control file, draws the prior ensemble, and runs PEST++ IES; rerun it after moving a slider.

`pyemu.ParameterEnsemble.from_gaussian_draw()` draws `num_reals` parameter sets around the means, with standard deviations from the bounds, in log space because the parameters are log transformed, and `enforce()` clips any draw outside its bounds. The draw has its own random number generator with a fixed seed (`rng`), so a rerun draws the same prior and only the sliders change the result. The observations have weights but no other noise options, so IES matches every realization to the observed heads themselves, without added noise. In each iteration, up to `noptmax`, IES moves every realization toward the observations, using sensitivities estimated from the ensemble itself, and drops realizations whose phi is more than 1.7 standard deviations worse than the mean (`ies_bad_phi_sigma`). `start_workers()` runs the realizations in parallel on `num_workers` workers. PEST++ IES normally gives up on a run that takes much longer than the average and counts it as failed, which on a busy computer can happen to a run that is merely slow; `overdue_giveup_fac` is set high enough that no run is given up.

In [ ]:
num_reals = 50
num_workers = 25
noptmax = 5

In [ ]:
# the final values of the sliders: the bounds, then the mean inside them
for name in ["kupper", "klower", "kv", "rch"]:
    par.loc[name, "parlbnd"] = controls[f"{name}_low"].value
    par.loc[name, "parubnd"] = controls[f"{name}_high"].value
    par.loc[name, "parval1"] = np.clip(
        controls[f"{name}_mean"].value, par.parlbnd[name], par.parubnd[name]
    )
obs.loc[obs.obgnme == "wt", "weight"] = controls["wt_weight"].value
obs.loc[obs.obgnme == "aq", "weight"] = controls["aq_weight"].value

# pe = pyemu.ParameterEnsemble.from_gaussian_draw(
#     pst, num_reals=num_reals, rng=np.random.RandomState(11111)
# )
# pe.enforce()
# pe.to_csv(ws / "prior_pe.csv")

# pst.pestpp_options["ies_parameter_ensemble"] = "prior_pe.csv"
pst.pestpp_options["ies_num_reals"] = num_reals
pst.pestpp_options["ies_bad_phi_sigma"] = 1.7
pst.pestpp_options.pop("ies_observation_ensemble", None)  # no noise ensemble
pst.pestpp_options["overdue_giveup_fac"] = 1.0e30
pst.control_data.noptmax = noptmax
pst.write(ws / "pest.pst", version=2)

m_d = pl.Path("models/sv-steady-ies")
pyemu.os_utils.start_workers(
    str(ws),
    "pestpp-ies",
    "pest.pst",
    num_workers=num_workers,
    worker_root="models",
    master_dir=str(m_d),
    verbose=False,
)
phi = pd.read_csv(m_d / "pest.phi.actual.csv", index_col=0)
print(phi[["mean", "min", "max"]].round(1))

## Prior and posterior

The left plot shows the phi of every realization at each iteration. The two plots beside it show each realization's recharge multiplier against the K above and below the confining unit, before (gray) and after (blue) history matching. The table gives the prior and posterior ranges of the parameters.

In [ ]:
pst = pyemu.Pst(os.path.join(m_d, "pest.pst"))
noptmax = pst.control_data.noptmax
prior_pe = pd.read_csv(m_d / "pest.0.par.csv", index_col=0)
post_pe = pd.read_csv(
    m_d / f"pest.{pst.ies.phiactual.iteration.max()}.par.csv", index_col=0
)
prior_oe = pd.read_csv(m_d / "pest.0.obs.csv", index_col=0).loc[:, obs.obsnme]
post_oe = pd.read_csv(
    m_d / f"pest.{pst.ies.phiactual.iteration.max()}.obs.csv", index_col=0
).loc[:, obs.obsnme]
reals = phi.columns[5:]  # the realization columns follow the summary columns

fig, axd = plt.subplot_mosaic(
    [["A", "B", "C"]], figsize=(12.0, 4.0), layout="constrained"
)
ax = axd["A"]
ax.plot(phi.index, phi.loc[:, reals], color="0.6", lw=0.5)
ax.plot(phi.index, phi["mean"], color="black", lw=1.5, label="Ensemble mean")
ax.axhline(pst.nnz_obs, color="red", lw=1.0, label="Number of observations")
ax.set_yscale("log")
ax.set_xlabel("Iteration")
ax.set_ylabel("Phi")
ax.legend(loc="upper right")

ax = axd["B"]
ax.plot(prior_pe.kupper, prior_pe.rch, "o", color="0.7", ms=4, label="Prior")
ax.plot(post_pe.kupper, post_pe.rch, "o", color="tab:blue", ms=4, label="Posterior")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("K above the confining unit (m/d)")
ax.set_ylabel("Recharge multiplier")
ax.legend(loc="lower right")

ax = axd["C"]
ax.plot(prior_pe.klower, prior_pe.rch, "o", color="0.7", ms=4)
ax.plot(post_pe.klower, post_pe.rch, "o", color="tab:blue", ms=4)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("K below the confining unit (m/d)")
ax.set_ylabel("Recharge multiplier")

plt.show()

pd.DataFrame(
    {
        "prior 5%": prior_pe.quantile(0.05),
        "posterior 5%": post_pe.quantile(0.05),
        "posterior median": post_pe.median(),
        "posterior 95%": post_pe.quantile(0.95),
        "prior 95%": prior_pe.quantile(0.95),
    }
).loc[["kupper", "klower", "kv", "rch"]]

## One-to-one plot

The heads of every realization against the observed heads, one small dot for each head of each realization: the prior in gray, the posterior in blue, and in red the values each realization was matched to, `pest.obs+noise.csv`. Without noise these are the observed heads themselves, so the red dots sit on the line. The larger blue dots are the posterior medians, and the box gives the statistics of their residuals; its phi uses a weight of 10 on every head, whatever the weight sliders chose, so that it compares with the phi of the steady-state heads in the other notebooks.

In [ ]:
observed = obs.obsval[heads]
median = post_oe.loc[:, observed.index].median()
residual = observed - median
low = observed.min() - 1.0
high = observed.max() + 1.0

obs_noise = pd.read_csv(m_d / "pest.obs+noise.csv", index_col=0)

fig, ax = plt.subplots(figsize=(6.0, 6.0), layout="constrained")
ax.plot([low, high], [low, high], color="black", lw=1.0)
for real in prior_oe.index:
    ax.plot(observed, prior_oe.loc[real, observed.index], ".", color="0.6", ms=2)
for real in obs_noise.index:
    ax.plot(observed, obs_noise.loc[real, observed.index], ".", color="red", ms=2)
for real in post_oe.index:
    ax.plot(observed, post_oe.loc[real, observed.index], ".", color="tab:blue", ms=2)
ax.plot([], [], ".", color="0.6", label="Prior")
ax.plot([], [], ".", color="red", label="Observed + noise")
ax.plot([], [], ".", color="tab:blue", label="Posterior")
ax.plot(observed, median, "o", color="tab:blue", ms=5, label="Posterior median")
ax.set_xlim(low, high)
ax.set_ylim(low, high)
ax.set_aspect("equal")
ax.set_xlabel("Observed head (m)")
ax.set_ylabel("Simulated head (m)")
ax.legend(loc="lower right")
weighted = 10.0 * residual  # weight 10 for the steady-state heads, as in c, f, g
stats = (
    f"observations {len(residual)}\n"
    f"mean residual {residual.mean():.3f} m\n"
    f"RMS residual {np.sqrt((residual**2).mean()):.3f} m\n"
    f"largest |residual| {residual.abs().max():.3f} m\n"
    f"phi of the median {(weighted**2).sum():.1f} (weight 10)"
)
ax.text(0.03, 0.97, stats, transform=ax.transAxes, va="top", family="monospace")
plt.show()

## The prediction

The steady-state discharge of groundwater to the stream, the negative of `riv-swgw`, in every realization of the prior (gray) and the posterior (blue), with the truth as the red line.

In [ ]:
# the same bins for every histogram, so a narrow posterior shows as a tall bar
bins = np.linspace(-prior_oe[prediction].max(), -prior_oe[prediction].min(), 40)
fig, ax = plt.subplots(figsize=(7.0, 3.5), layout="constrained")
ax.hist(-prior_oe[prediction], bins=bins, color="0.7", alpha=0.7, label="Prior")
ax.hist(-post_oe[prediction], bins=bins, color="tab:blue", alpha=0.7, label="Posterior")
ax.axvline(-obs.obsval[prediction], color="red", lw=1.5, label="Truth")
ax.set_xlabel("Steady-state discharge to the stream (m$^3$/d)")
ax.set_ylabel("Realizations")
ax.legend(loc="upper right")
plt.show()